# LLM APIs, Reasoning Flows & Prompt-to-Action Pipelines

**Track:** Applied Agent Engineering Foundations  
**Module:** LLM Foundations  
**Environment:** Local VS Code / Jupyter Notebook consuming Azure-hosted model endpoints  
**Audience:** Software engineers building enterprise AI workflows with Azure-hosted LLMs

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect securely from a local notebook to an approved Azure-hosted model endpoint.
2. Read approved model deployment names from environment configuration.
3. Use Azure/OpenAI-compatible LLM APIs for prompt experiments and reasoning flows.
4. Inspect model responses, token usage, and response metadata.
5. Build reusable LLM helper functions.
6. Capture prompt/response telemetry for observability and cost analysis.
7. Build a model-agnostic prompt-to-action pipeline with a controlled backend lookup.

## Before you run

This notebook assumes:
- you are running **VS Code / Jupyter locally**, outside the Azure platform
- Python 3.10 and the required packages are installed
- the `.env` file is available in the project root
- the `.env` file contains the approved Azure endpoint, API key, and deployment names
- the Azure-hosted model deployments are already created and accessible
- `training_user_directory.csv` is available locally for the backend lookup exercise

### Required `.env` variables

```text
AZURE_OPENAI_ENDPOINT=<approved Azure endpoint ending in /openai/v1/>
AZURE_OPENAI_API_KEY=<approved API key>
AZURE_OPENAI_MODEL=<primary chat deployment name>
AZURE_OPENAI_EMBEDDING_MODEL=<embedding deployment name>
AZURE_OPENAI_MODEL_SECONDARY=<optional secondary chat deployment name>
```

### Design choice for this lab
This notebook teaches a **model-agnostic prompt-to-action pipeline**.  
The learning flow remains: prompt → model decision → structured plan → validation → controlled backend action.

In [1]:
# Uncomment only if your environment is missing any package
# %pip install -q openai python-dotenv pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load configuration from the project-level .env file.
# If this notebook is stored inside a notebooks/ folder, ../.env is typically correct.
load_dotenv("../.env", override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL = os.getenv("AZURE_OPENAI_MODEL")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_MODEL_SECONDARY = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")

if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_API_KEY or not AZURE_OPENAI_MODEL:
    raise ValueError(
        "Missing required Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, and AZURE_OPENAI_MODEL in ../.env"
    )

client = OpenAI(
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY
)

print("Azure endpoint configured:", bool(AZURE_OPENAI_ENDPOINT))
print("API key configured       :", bool(AZURE_OPENAI_API_KEY))
print("Primary model deployment :", AZURE_OPENAI_MODEL)
print("Embedding deployment     :", AZURE_OPENAI_EMBEDDING_MODEL)
print("Secondary deployment     :", AZURE_OPENAI_MODEL_SECONDARY)

Azure endpoint configured: True
API key configured       : True
Primary model deployment : gpt-4.1-mini
Embedding deployment     : text-embedding-3-small
Secondary deployment     : gpt-5-mini


## Step 1 — Show configured Azure model deployments

**Goal:** Let learners see which approved Azure deployments this local notebook is configured to consume.

**Discuss:**
- Azure model deployments are created and governed centrally
- deployment names should come from configuration rather than being hardcoded throughout the notebook
- different deployments can be used for chat, embeddings, or model comparison
- local code can consume enterprise-hosted Azure endpoints without running inside Azure

In [2]:
def list_configured_models() -> pd.DataFrame:
    rows = [
        {"purpose": "Primary chat / generation", "deployment": AZURE_OPENAI_MODEL},
        {"purpose": "Secondary chat / comparison", "deployment": AZURE_OPENAI_MODEL_SECONDARY},
        {"purpose": "Embeddings", "deployment": AZURE_OPENAI_EMBEDDING_MODEL},
    ]
    return pd.DataFrame(rows)

models_df = list_configured_models()
display(models_df)

,purpose,deployment
0,Primary chat / generation,gpt-4.1-mini
1,Secondary chat / comparison,gpt-5-mini
2,Embeddings,text-embedding-3-small


## Step 2 — Secure endpoint usage and minimal smoke test

**Goal:** Confirm that the local Jupyter notebook can call the approved Azure model endpoint safely.

**Secure usage principles:**
- do **not** hardcode API keys in notebooks
- load credentials and deployment names from `.env`
- keep `.env` outside source control
- use only approved Azure model deployments
- start with a **small, cheap** smoke test
- fail early if required configuration is missing

**Why this matters:**  
In enterprise settings, configuration and access controls should be validated **before** a costly or unsafe request is sent.

In [3]:
# Use the primary Azure chat deployment configured in .env
AZURE_TEXT_MODEL = AZURE_OPENAI_MODEL

print("Default text model:", AZURE_TEXT_MODEL)
print("Endpoint:", AZURE_OPENAI_ENDPOINT)

Default text model: gpt-4.1-mini
Endpoint: https://genfoundry.cognitiveservices.azure.com/openai/v1/


In [4]:
# Call the Azure-hosted LLM and get a response
response = client.chat.completions.create(
    model=AZURE_TEXT_MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful enterprise AI assistant."
        },
        {
            "role": "user",
            "content": "In two lines, summarize the plot of the movie Inception."
        }
    ],
    max_tokens=250,
    temperature=0.5,
    top_p=0.9
)

print(response)

ChatCompletion(id='chatcmpl-EBfudjaV1XHXlLdrNDkbW9DLGwZiM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Inception follows a skilled thief who enters people's dreams to steal secrets, undertaking a final mission to implant an idea into a target's subconscious. As layers of dreams unfold, he struggles to distinguish reality from illusion.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1786451427, model='gpt-4.1-mini-2025-04-14', object='chat.completion', moderation=None, service_tier='default', system_fingerpri

In [5]:
print(response.usage)

CompletionUsage(completion_tokens=43, prompt_tokens=32, total_tokens=75, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 10, 'engine_ttft_ms': 37, 'engine_ttlt_ms': 450, 'pre_inference_ms': 162, 'service_tbt_ms': 10, 'service_ttft_ms': 395, 'service_ttlt_ms': 802, 'total_duration_ms': 656, 'user_visible_ttft_ms': 233})


In [33]:
# Now do the same with a helper function that we can reuse for the rest of the notebook
def ask_llm(user_prompt: str,
            system_prompt: str = "You are a helpful enterprise AI assistant.",
            model_id: str = AZURE_TEXT_MODEL,
            max_tokens: int = 250,
            temperature: float = 0.2,
            top_p: float = 0.9) -> str:

    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )

    return response.choices[0].message.content

In [10]:
# Test the helper function with a new prompt
print(ask_llm("In two lines, explain what an LLM API call is."))

An LLM API call is a request sent to a Large Language Model's interface to generate text-based responses or perform language tasks. It allows developers to integrate the model's capabilities into applications by providing input prompts and receiving generated outputs.


In [34]:
# Change the parameters to see how the response changes
print(ask_llm("In two lines, explain what an LLM API call is.",max_tokens=100, temperature=0.5))

An LLM API call is a request sent to a Large Language Model's interface to generate text, answer questions, or perform language-related tasks. It enables applications to leverage the model's capabilities by providing input prompts and receiving generated responses programmatically.


In [35]:
# Activity: Change prompt parameters
# Task:
# Try the same prompt with:
# 1. temperature = 0
# 2. temperature = 0.7
# 3. max_tokens = 50
# 4. top_p = 0.5
# Compare the outputs.

prompt = "Explain why LLM validation matters in enterprise applications."

In [ ]:
# Activity: 
# Ask the model about a very recent or real-time event.
# Example:
# - "What are the latest news updates today?"
# - "Who won yesterday's IPL match?"
# - "What is happening in global markets right now?"


In [13]:
print(ask_llm("Who won yesterday's IPL match?",max_tokens=100, temperature=0.5))

I don't have access to real-time information. To find out who won yesterday's IPL match, please check the latest sports news on a reliable website, the official IPL app, or a sports news channel. If you provide me with the teams that played, I might be able to help with historical data!


# LLM Interaction Logging & Observability Framework

Build a reusable LLM interface that not only generates responses but also captures structured telemetry for monitoring, cost tracking, and analysis.

In [21]:
import os
import pandas as pd
from datetime import datetime

LOG_CSV_PATH = "llm_prompt_log.csv"

def ask_llm(
    user_prompt: str,
    asked_by: str,
    system_prompt: str = "You are a helpful enterprise AI assistant.",
    model_id: str = AZURE_TEXT_MODEL,
    max_tokens: int = 250,
    temperature: float = 0.2,
    csv_path: str = LOG_CSV_PATH
) -> str:
    # Call the Azure-hosted model, return the answer,
    # and log prompt/response details into a CSV using pandas.

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )

    answer = response.choices[0].message.content

    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else None
    output_tokens = usage.completion_tokens if usage else None
    total_tokens = usage.total_tokens if usage else None

    new_row = pd.DataFrame([{
        "timestamp": timestamp,
        "asked_by": asked_by,
        "model_id": model_id,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response_text": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "max_tokens": max_tokens,
        "temperature": temperature
    }])

    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        updated_df = pd.concat([existing_df, new_row], ignore_index=True)
    else:
        updated_df = new_row

    updated_df.to_csv(csv_path, index=False)

    return answer

In [22]:
user_prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
user = "user@123"
ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

'Validation matters in enterprise AI systems because it ensures the accuracy, reliability, and trustworthiness of AI models before deployment. Proper validation helps identify and mitigate biases, errors, and performance issues, reducing risks of incorrect decisions that could lead to financial loss, reputational damage, or regulatory non-compliance. It also supports compliance with industry standards and legal requirements, fosters user confidence, and enables continuous improvement by providing feedback on model behavior in real-world scenarios. Overall, validation is critical for maintaining the integrity and effectiveness of AI solutions in complex enterprise environments.'

In [23]:
# Activity:
# Please try asking the model a few more questions with different system prompts and parameters to generate more rows in the log for analysis.
# Inspect the LLM log
# Task:
# Display the CSV log and compare token usage across prompts.

# user_prompt = ""
# system_prompt = ""
# user = ""
# ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

log_df = pd.read_csv(LOG_CSV_PATH)
display(log_df)

,timestamp,asked_by,model_id,system_prompt,user_prompt,response_text,input_tokens,output_tokens,total_tokens,max_tokens,temperature
0,2026-08-11 18:04:10,user@123,gpt-4.1-mini,You are an enterprise AI agent. \n ...,Summarize why validation matters in enterprise...,Validation matters in enterprise AI systems be...,40,109,149,250,0.2


## Reflection checkpoint
Discuss pros and cons of prompt style before move to next section

## Step 4 — Build a prompt-to-action pipeline

A production-safe workflow often looks like this:
1. understand the request
2. convert it into a structured plan
3. validate the plan
4. execute only the allowed action
5. call a controlled backend function
6. log the result

This is the core **tool/function calling pattern** we want learners to understand.

### Important note
This notebook uses a **model-agnostic structured JSON plan**.  
That keeps the concept simple and portable across models.

In [24]:
ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_backend_action": true or false,
  "action_name": "<action or none>",
  "arguments": { ... },
  "reason_for_action": "<one sentence>"
}

Allowed action names:
- create_ticket
- lookup_user_record
- none

Examples:
User: "Create an IT ticket for VPN access issue"
Output:
{"intent":"create_it_ticket","needs_backend_action":true,"action_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_action":"A ticket must be created in the backend system."}

User: "Find the team and location for ananya"
Output:
{"intent":"lookup_user_record","needs_backend_action":true,"action_name":"lookup_user_record","arguments":{"user_name":"ananya"},"reason_for_action":"The answer should come from the approved backend CSV data source."}
'''

In [25]:
# Function to extract JSON object from LLM response
def extract_json_object(text: str) -> dict:
    return json.loads(text.strip())

# Function to plan action based on user request
def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_prompt=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_prompt="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0,
        asked_by = "user@123"
    )
    return extract_json_object(raw)

In [26]:
# Test the action planner with a sample user request
plan_action("Create an IT ticket for laptop replacement")

{'intent': 'create_it_ticket',
 'needs_backend_action': True,
 'action_name': 'create_ticket',
 'arguments': {'category': 'IT Support', 'summary': 'laptop replacement'},
 'reason_for_action': 'A ticket must be created in the backend system.'}

In [27]:
# Test the action planner with another sample user request
plan_action("Find the team and location for ananya")

{'intent': 'lookup_user_record',
 'needs_backend_action': True,
 'action_name': 'lookup_user_record',
 'arguments': {'user_name': 'ananya'},
 'reason_for_action': 'The answer should come from the approved backend CSV data source.'}

In [28]:
import pandas as pd

# Read local backend data to be used for action execution.
# Keep training_user_directory.csv in the notebook working folder,
# or update LOCAL_DATA_FILE to the appropriate relative path.
LOCAL_DATA_FILE = "training_user_directory.csv"

def read_csv_from_local(file_path: str) -> pd.DataFrame:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"{file_path} was not found. Place the training CSV in the notebook "
            "working folder or update LOCAL_DATA_FILE."
        )
    return pd.read_csv(file_path)

df = read_csv_from_local(LOCAL_DATA_FILE)
print(df.head())

  user_name    full_name                  team   location      manager  \
0    ananya   Ananya Rao      Data Engineering  Bengaluru  Rahul Mehta   
1    vikram  Vikram Shah  Platform Engineering  Hyderabad   Priya Nair   
2      neha   Neha Gupta         QA Automation       Pune  Sandeep Rao   
3     arjun   Arjun Iyer            IT Support    Chennai  Meera Joshi   
4      sara    Sara Khan               Product  Hyderabad    Dev Malik   

     laptop_type  
0    MacBook Air  
1   ThinkPad T14  
2    MacBook Pro  
3    ThinkPad X1  
4  Dell Latitude  


In [29]:
# Function to execute the planned action
def execute_plan(plan: dict) -> dict:
    action_name = plan["action_name"]

    if action_name == "lookup_user_record":
        user_name = plan["arguments"]["user_name"]
        df = read_csv_from_local(LOCAL_DATA_FILE)

        match = df[df["user_name"].str.lower() == user_name.lower()]

        if match.empty:
            return {
                "status": "not_found",
                "message": f"No user found for '{user_name}'."
            }

        row = match.iloc[0]

        return {
            "status": "success",
            "message": "User found.",
            "user_record": {
                "user_name": row["user_name"],
                "full_name": row["full_name"],
                "team": row["team"],
                "location": row["location"],
                "manager": row["manager"],
                "laptop_type": row["laptop_type"]
            }
        }

    return {
        "status": "no_action",
        "message": "No backend action required."
    }

In [30]:
# run the full pipeline with a sample user request
def run_prompt_to_action_pipeline(user_request: str) -> dict:
    plan = plan_action(user_request)
    result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "result": result
    }

In [31]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for ananya")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for ananya",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "ananya"
    },
    "reason_for_action": "The answer should come from the approved backend CSV data source."
  },
  "result": {
    "status": "success",
    "message": "User found.",
    "user_record": {
      "user_name": "ananya",
      "full_name": "Ananya Rao",
      "team": "Data Engineering",
      "location": "Bengaluru",
      "manager": "Rahul Mehta",
      "laptop_type": "MacBook Air"
    }
  }
}


In [32]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for karthik")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for karthik",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "karthik"
    },
    "reason_for_action": "The answer should come from the approved backend CSV data source."
  },
  "result": {
    "status": "not_found",
    "message": "No user found for 'karthik'."
  }
}


## CRAFT+E Section

In [36]:
# using CRAFT+E Framework build "Build a Healthcare Policy Assistant"